# 함수 호출
- 대규모 언어 모델을 외부 도구에 연결하는 방법  

- API 호출에서 함수를 설명하고 모델이 하나 이상의 함수를 호출하기 위한 인수가 포함된 JSON 개체를 출력하도록 지능적으로 선택하도록 할 수 있습니다. Chat Completions API는 함수를 호출하지 않습니다. 대신 모델은 코드에서 함수를 호출하는 데 사용할 수 있는 JSON을 생성합니다.
- 모델은 함수를 호출해야 하는 시기를 감지하고 함수를 준수하는 JSON으로 응답하도록 훈련되었습니다. 이 기능에는 잠재적인 위험도 따릅니다. 사용자를 대신하여 세상에 영향을 미치는 조치(이메일 전송, 온라인 게시, 구매 등)를 수행하기 전에 사용자 확인 흐름을 구축하는 것이 좋습니다.

### 채팅 모델로 함수를 호출하는 방법

함수 호출을 위한 기본 단계는 다음과 같습니다:

1. **모델 호출**: 사용자 질의와 함께 함수 집합을 정의하여 모델을 호출합니다. 함수 목록을 `tools` 파라미터에 제공합니다.


2. **모델의 함수 호출 선택**: 모델은 하나 이상의 함수를 호출할 수 있습니다. 이 경우, 모델의 응답 내용은 사용자가 정의한 스키마에 따른 JSON 객체 문자열이 됩니다. 


3. **JSON 파싱 및 함수 호출**: 코드에서 문자열을 JSON으로 파싱하고, 제공된 인수가 있는 경우 함수를 호출합니다.


4. **모델 재호출 및 결과 요약**: 함수 응답을 새로운 메시지로 추가하여 모델을 다시 호출합니다. 모델이 결과를 사용자에게 요약하여 전달합니다.

이 단계들은 사용자 질의에 따라 적절한 함수를 선택하고, 해당 함수의 응답을 처리하여 최종 결과를 사용자에게 제공하는 과정을 포함합니다. 이를 통해 사용자는 더 나은 응답을 받을 수 있습니다.

### 함수 정의 (Defining Functions)
- 각 API 요청의 tools 매개변수 내에서 함수를 정의할 수 있습니다.
- 함수는 스키마(Schema) 를 통해 정의되며, 모델이 함수의 목적과 입력 인수를 이해할 수 있도록 돕습니다.
스키마는 다음과 같은 필드로 구성됩니다.
| 필드        | 설명                                      |
|------------|------------------------------------------|
| **name**   | 함수의 이름 (예: `get_weather`)          |
| **description** | 함수가 언제, 어떻게 사용되는지 설명  |
| **parameters**  | 함수의 입력 인수를 정의하는 JSON 스키마 |


In [1]:
from dotenv import load_dotenv
load_dotenv() 

True

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-mini"

In [3]:
# 함수 정의 추가
def get_fortune(zodiac):
    fortunes = {
        "쥐띠": "금전운이 좋습니다. 작은 투자로 큰 수익을 볼 수 있으니 신중하게 결정하세요.",
        "소띠": "건강에 유의하세요. 규칙적인 생활이 행운을 부릅니다.",
        "호랑이띠": "대인관계가 원활합니다. 새로운 인연을 만날 수 있는 좋은 날입니다.",
        "토끼띠": "다음 주 화요일에 귀인을 만나 좋은 소식을 듣게 될 것입니다.",
        "용띠": "직장에서 좋은 평가를 받을 수 있습니다. 자신감을 가지세요.",
        "뱀띠": "학업운이 상승합니다. 새로운 공부를 시작하기 좋은 시기입니다.",
        "말띠": "가족과의 시간이 행복을 가져다줍니다.",
        "양띠": "재물운이 들어옵니다. 하지만 충동구매는 자제하세요.",
        "원숭이띠": "창의력이 빛나는 시기입니다. 새로운 아이디어를 실행하세요.",
        "닭띠": "여행운이 좋습니다. 가까운 곳으로 떠나보세요.",
        "개띠": "연애운이 상승합니다. 용기를 내어 마음을 전해보세요.",
        "돼지띠": "건강운이 좋습니다. 새로운 운동을 시작하기 좋은 때입니다."
    }
    return fortunes.get(zodiac, f"{zodiac}: 오늘은 평온한 하루가 될 것입니다.")

# 예제 더미 함수로 하드 코딩된 동일한 날씨 정보를 반환합니다.
# 실제 환경에서는 백엔드 API 또는 외부 API가 될 수 있습니다.
def get_current_weather(location, unit="celsius"):
    """지정된 위치의 현재 날씨를 가져옵니다."""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "22.2", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

get_current_weather("Paris")

'{"location": "Paris", "temperature": "22", "unit": "celsius"}'

In [4]:
# 1. 모델에서 사용 가능한 도구 목록을 정의
tools = [
    {
        "type": "function",
        "name": "get_fortune",
        "description": "오늘의 띠별 운세를 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "zodiac": {
                    "type": "string",
                    "description": "십이지 띠 (예: 쥐띠, 소띠, 호랑이띠, 토끼띠, 용띠, 뱀띠, 말띠, 양띠, 원숭이띠, 닭띠, 개띠, 돼지띠)",
                },
            },
            "required": ["zodiac"],
        },
    },
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "지정된 위치의 현재 날씨를 가져오는 함수.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "도시와 주 이름. 예: San Francisco, CA"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "온도 단위"
                }
            },
            "required": ["location"]
        }
    }
]

input_list = [
    {"role": "user", "content": "제 오늘 운세 좀 봐주세요. 저는 토끼띠입니다."},
    {"role": "user", "content": "샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?"}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model=Model,
    tools=tools,
    input=input_list,
)

In [5]:
response.output

[ResponseReasoningItem(id='rs_0850e016ae8fb245006901843b239081959a2181c95cfe1ca7', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"zodiac":"토끼띠"}', call_id='call_76JGVg938JiTV8swmEhIkdqQ', name='get_fortune', type='function_call', id='fc_0850e016ae8fb245006901843ea4108195874058f8a13b0931', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"San Francisco, CA","unit":"celsius"}', call_id='call_ZhOhpOSh5S9aIlIpVt1MCicZ', name='get_current_weather', type='function_call', id='fc_0850e016ae8fb245006901843ee08c819591a2a3e171ec157d', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"Tokyo, Japan","unit":"celsius"}', call_id='call_D6hNUIyTUjViaQgQTnf8g9tv', name='get_current_weather', type='function_call', id='fc_0850e016ae8fb245006901843f2a1c8195a48ec246c94e59f8', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"Paris, France","unit":"celsius"}', call_id='call

In [6]:
# 이후 요청에 사용할 함수 호출 결과를 저장합니다
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_current_weather":
            # 3. get_current_weather 함수 실행
            args = json.loads(item.arguments)
            weather = get_current_weather(
                location=args.get("location"),
                unit=args.get("unit", "celsius")
            )
            
            # 4. 함수 실행 결과를 모델에 전달
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                    "weather": weather
                }, ensure_ascii=False)
            })
            
        elif item.name == "get_fortune":
            # 3. get_fortune 함수 실행
            args = json.loads(item.arguments)
            fortune = get_fortune(args["zodiac"])
            
            # 4. 함수 실행 결과를 모델에 전달
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                    "fortune": fortune
                }, ensure_ascii=False)
            })

print("최종 입력 내역:")
for i, item in enumerate(input_list):
    print(f"[{i}] {item}")

최종 입력 내역:
[0] {'role': 'user', 'content': '제 오늘 운세 좀 봐주세요. 저는 토끼띠입니다.'}
[1] {'role': 'user', 'content': '샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?'}
[2] ResponseReasoningItem(id='rs_0850e016ae8fb245006901843b239081959a2181c95cfe1ca7', summary=[], type='reasoning', content=None, encrypted_content=None, status=None)
[3] ResponseFunctionToolCall(arguments='{"zodiac":"토끼띠"}', call_id='call_76JGVg938JiTV8swmEhIkdqQ', name='get_fortune', type='function_call', id='fc_0850e016ae8fb245006901843ea4108195874058f8a13b0931', status='completed')
[4] ResponseFunctionToolCall(arguments='{"location":"San Francisco, CA","unit":"celsius"}', call_id='call_ZhOhpOSh5S9aIlIpVt1MCicZ', name='get_current_weather', type='function_call', id='fc_0850e016ae8fb245006901843ee08c819591a2a3e171ec157d', status='completed')
[5] ResponseFunctionToolCall(arguments='{"location":"Tokyo, Japan","unit":"celsius"}', call_id='call_D6hNUIyTUjViaQgQTnf8g9tv', name='get_current_weather', type='function_call', id='fc_0850e016ae8fb245006901843f2a1

In [7]:
response = client.responses.create(
    model=Model,
    instructions="도구로 생성된 운세와 날씨 정보를 사용하여 친절하게 응답하세요.",
    tools=tools,
    input=input_list,
)

# 5. 모델이 최종 응답을 생성합니다!
print("\n최종 출력:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)


최종 출력:
{
  "id": "resp_0850e016ae8fb245006901844ca78c81958ec5e42ca8d4cebf",
  "created_at": 1761707084.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "도구로 생성된 운세와 날씨 정보를 사용하여 친절하게 응답하세요.",
  "metadata": {},
  "model": "gpt-5-mini-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "rs_0850e016ae8fb245006901844d7e1881959947f978a6b5b584",
      "summary": [],
      "type": "reasoning",
      "content": null,
      "encrypted_content": null,
      "status": null
    },
    {
      "id": "msg_0850e016ae8fb2450069018453f2ac8195b0cc45f723238477",
      "content": [
        {
          "annotations": [],
          "text": "알겠습니다 — 요청하신 토끼띠 운세와 세 도시의 현재 날씨입니다.\n\n운세 (토끼띠)\n- 다음 주 화요일에 귀인을 만나 좋은 소식을 듣게 될 것입니다. 지금은 차분히 준비하시고, 기회가 왔을 때 잘 잡으시면 좋겠습니다.\n\n현재 날씨 (단위: 섭씨)\n- 샌프란시스코: 22.2°C — 온화한 편입니다. 가벼운 겉옷이나 긴팔 하나면 무난합니다.\n- 도쿄: 10°C — 쌀쌀합니다. 코트나 두꺼운 겉옷, 목도리 등을 추천합니다.\n- 파리: 22°C — 따뜻한 편입니다. 반팔이나 가벼운 긴팔로 괜찮습니다.\n\n원하시면 온도를 화씨로 변환해 드리거나, 시간대별/일기예보(며칠간)도 알려드릴게요."

### Free Weather API 이용
https://open-meteo.com/

In [8]:
import requests

# 현재의 온도 가져오기
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = response.json()
    return data['current']['temperature_2m']

# 서울의 위도, 경도
get_weather(37.56667, 126.97806)

13.1

In [9]:
input_messages = [{"role": "user", "content": "오늘 서울의 기온이 어때?"}]

tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "제공된 좌표의 현재 기온을 섭씨(Celsius) 단위로 가져오세요.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True   # JSON 스키마의 엄격한 준수(strict mode)를 적용
    }]

response = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)

response.output

[ResponseReasoningItem(id='rs_042424589bd59ade0069018459d3688195ad446a1821a7d5a0', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"latitude":37.5665,"longitude":126.9780}', call_id='call_9NQ50RP6dSaFEbEXlNjh1RGd', name='get_weather', type='function_call', id='fc_042424589bd59ade006901845c38048195bba3960ceecaf294', status='completed')]

In [10]:
# 이후 요청에 사용할 함수 호출 결과 저장
input_messages += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_weather":
            args = json.loads(item.arguments)
            result = get_weather(args['latitude'], args['longitude'])

            input_messages.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps({
                        "weather": result
                    }, ensure_ascii=False)
                })
            
result

13.1

In [11]:
response_2 = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)

print(response_2.output_text)

현재 서울의 기온은 약 13.1°C입니다.  
원하시면 오늘 하루 예상 최고/최저 기온이나 강수 예보도 알려드릴게요.
